# COS30049 A2 - Round 2 data for the 152M model's chat and explanation stages (Kaggle CPU session)

Tokenizes, with the 152M model's own tokenizer, what its two fine-tuning stages train on (`docs/KAGGLE_LM_PLAN.md`,
section 4): DS6 smol-smoltalk for the chat stage, as in round 1, and DS7 v2 for the explanation stage, with the
glossary's training rows repeated 4 times as for Qwen v2. CPU only, so no GPU quota.

**Settings:** Accelerator **None**, **Internet on**. **Inputs:** this account's dataset `cos30049a2-text150m-base`
(the tokenizer) and notebook `cos30049a2-qwen-data` (DS7 v2). Output `stage/` and `stage_manifest.json`, read by
`cos30049a2-text150m-stages`.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "11973ad51a741ee4051b28b90a4e7d5a6295a767"   # the code this build uses; recorded in stage_manifest.json
PINS = ["numpy==2.2.6", "tokenizers==0.21.1", "huggingface_hub==0.30.2", "pyarrow==20.0.0"]  # environment.yml
GLOSSARY_REPEATS = 4   # as configs/qwen3b_explain_v2.json -> repeat_sources

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

def find_input(rel):
    """The file under /kaggle/input whose path ends with rel (inputs mount under their own folder names)."""
    hits = [p for p in Path("/kaggle/input").rglob(Path(rel).name) if p.as_posix().endswith(rel)]
    if not hits:
        raise SystemExit(f"{rel} is not in any input: add the inputs named at the top")
    return hits[0]


def check_base():
    """The round-1 files (dataset cos30049a2-text150m-base), each checked against the dataset's SHA256SUMS.txt."""
    sums = find_input("SHA256SUMS.txt")
    for line in sums.read_text().splitlines():
        digest, name = line.split(maxsplit=1)
        assert sha256(sums.parent / name.lstrip("*")) == digest, f"{name} differs from SHA256SUMS.txt"
    return sums.parent

In [ ]:
base = check_base()
TOK = base / "tokenizer.json"
qwen_manifest_path = find_input("data_manifest.json")
qwen_manifest = json.loads(qwen_manifest_path.read_text(encoding="utf-8"))
DS7 = SRC / "data" / "processed" / "ds7_v2"
DS7.mkdir(parents=True, exist_ok=True)
for name in ("train.jsonl", "val.jsonl", "glossary_test.jsonl"):
    rel = f"data/processed/ds7_v2/{name}"
    shutil.copy(qwen_manifest_path.parent / rel, DS7 / name)
    assert sha256(DS7 / name) == qwen_manifest["files"][rel], f"{rel} differs from data_manifest.json"
# The glossary rows are in train.jsonl once; GLOSSARY_REPEATS - 1 more copies go in as extra files.
glossary = Path("/tmp/glossary_train.jsonl")
with open(DS7 / "train.jsonl", encoding="utf-8") as f, open(glossary, "w", encoding="utf-8") as out:
    out.writelines(line for line in f if json.loads(line)["source"] == "glossary")
STAGE = WORK / "stage"
sh(sys.executable, "-m", "lm.prepare_chat", "--tokenizer", str(TOK), "--out", str(STAGE / "chat"), log="prepare-chat.log")
sh(sys.executable, "-m", "lm.prepare_chat", "--tokenizer", str(TOK), "--source", "jsonl", "--jsonl", str(DS7 / "train.jsonl"),
   "--val-jsonl", str(DS7 / "val.jsonl"), "--extra", *[str(glossary)] * (GLOSSARY_REPEATS - 1),
   "--out", str(STAGE / "ds7_v2_chat"), log="prepare-ds7-v2-chat.log")
shutil.copy(DS7 / "glossary_test.jsonl", STAGE / "glossary_test.jsonl")

In [ ]:
manifest = {"commit": COMMIT, "ds7_v2_commit": qwen_manifest["commit"],
            "built_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "files": {p.relative_to(WORK).as_posix(): sha256(p) for p in sorted(STAGE.rglob("*")) if p.is_file()}}
(WORK / "stage_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 2**20:9.1f} MB  {p.relative_to(WORK)}")